# SU(2) gate diffusion experiments

Scratch notebook for running package experiments against Colab GPU or a local VS Code/Jupyter kernel. Keep project logic in `su2diffusion`; use this notebook to choose configs, run reports, and compare outputs.

In [ ]:
# Run this in Colab or any fresh remote runtime before importing su2diffusion.
# Change BRANCH when testing a PR branch; use "main" after merge.
BRANCH = "codex/two-entangler-benchmark"

!pip install --no-cache-dir --force-reinstall --no-deps git+https://github.com/joe-singh/su2diffusion.git@{BRANCH}

In [ ]:
from dataclasses import replace

import torch

from su2diffusion import (
    center_names_for_config,
    diagnose_samples,
    get_experiment_config,
    plot_experiment_report,
    plot_hidden_shallow_circuit_best_fidelities,
    plot_hidden_two_entangler_best_fidelities,
    plot_synthesis_fidelity_histograms,
    print_center_mass_table,
    print_conditional_label_table,
    print_diagnostics_table,
    print_per_center_table,
    print_synthesis_candidates,
    print_hidden_shallow_circuit_benchmark,
    print_hidden_shallow_circuit_summary,
    print_hidden_two_entangler_circuit_benchmark,
    print_hidden_two_entangler_circuit_summary,
    print_synthesis_summary,
    resample_experiment,
    run_hidden_shallow_circuit_benchmark,
    run_hidden_two_entangler_circuit_benchmark,
    run_experiment,
    slot_labels_for_named_target,
    synthesize_bell_state,
    synthesize_bell_state_report,
    synthesize_named_gate,
    synthesize_named_gate_label_grid,
    synthesize_named_gate_label_grid_report,
    synthesize_named_gate_report,
    synthesize_named_gate_unconstrained,
    synthesize_named_gate_unconstrained_report,
)
from su2diffusion.data import centers_for_config

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

## Train single-qubit Clifford diffusion

Run the configured experiment and inspect single-gate diagnostics before using the generated gates for synthesis.

In [ ]:
CONFIG_NAME = "baseline-clifford-cond"
EVAL_COUNT = 1000

config = get_experiment_config(CONFIG_NAME)
config = replace(config, sample_count=EVAL_COUNT, reference_count=EVAL_COUNT)
config

In [ ]:
result = run_experiment(config, device=device)

center_names = center_names_for_config(result.config.data)
print_diagnostics_table(result.diagnostics)
print()
print_center_mass_table(result.diagnostics, center_names=center_names)
if result.conditional_diagnostics is not None:
    print()
    print_conditional_label_table(result.conditional_diagnostics, center_names=center_names)
plot_experiment_report(result)

## Named 2-qubit synthesis sanity check

Use generated single-qubit gates as local layers around fixed entanglers for CZ, CNOT, and Bell-state preparation.

In [ ]:
# 2-qubit synthesis demo: generated local gates inside a local-entangler-local template.
local_gates = result.generated_stochastic
local_labels = [center_names[int(i)] for i in result.stochastic_labels]

print("Guided decomposition sanity check")
cz_report = synthesize_named_gate_report(
    local_gates,
    target="cz",
    entangler="cz",
    n_candidates=2000,
    top_k=5,
    local_labels=local_labels,
    slot_label_names=slot_labels_for_named_target("cz", "cz"),
    name="CZ guided",
)
cnot_report = synthesize_named_gate_report(
    local_gates,
    target="cnot",
    entangler="cz",
    n_candidates=2000,
    top_k=5,
    local_labels=local_labels,
    slot_label_names=slot_labels_for_named_target("cnot", "cz"),
    name="CNOT guided",
)
bell_report = synthesize_bell_state_report(
    local_gates,
    entangler="cnot",
    n_candidates=2000,
    top_k=5,
    local_labels=local_labels,
    slot_label_names=("I", "I", "H", "I"),
    name="Bell guided",
)

guided_reports = [cz_report, cnot_report, bell_report]
for title, report in zip(
    ["CZ target via CZ entangler", "CNOT target via CZ entangler", "Bell-state preparation via CNOT entangler"],
    guided_reports,
):
    print(f"\n{title}")
    print_synthesis_candidates(report.candidates)
print("\nGuided summary")
print_synthesis_summary(guided_reports)

print("\nLabel-grid synthesis search over one generated representative per Clifford label")
label_grid_reports = {
    "CZ label grid": synthesize_named_gate_label_grid_report(
        local_gates,
        local_labels,
        target="cz",
        entangler="cz",
        top_k=5,
        name="CZ label grid",
    ),
    "CNOT label grid": synthesize_named_gate_label_grid_report(
        local_gates,
        local_labels,
        target="cnot",
        entangler="cz",
        top_k=5,
        name="CNOT label grid",
    ),
}
for report in label_grid_reports.values():
    print(f"\n{report.name}")
    print_synthesis_candidates(report.candidates)
print("\nLabel-grid summary")
print_synthesis_summary(label_grid_reports)
plot_synthesis_fidelity_histograms(label_grid_reports, bins=60)

print("\nRandom generated-gate search")
random_reports = {
    "CZ random": synthesize_named_gate_unconstrained_report(
        local_gates,
        target="cz",
        entangler="cz",
        n_candidates=100_000,
        top_k=5,
        local_labels=local_labels,
        seed=1,
        name="CZ random",
    ),
    "CNOT random": synthesize_named_gate_unconstrained_report(
        local_gates,
        target="cnot",
        entangler="cz",
        n_candidates=100_000,
        top_k=5,
        local_labels=local_labels,
        seed=2,
        name="CNOT random",
    ),
}
for report in random_reports.values():
    print(f"\n{report.name}")
    print_synthesis_candidates(report.candidates)
print("\nRandom search summary")
print_synthesis_summary(random_reports)
plot_synthesis_fidelity_histograms(random_reports, bins=60)

## Hidden shallow-circuit benchmark

Generate shallow 2-qubit targets from exact Clifford local layers, hide the labels, and recover high-fidelity circuits with generated local gates.

In [ ]:
# Hidden shallow-circuit benchmark: generate targets from exact Clifford local layers,
# hide the local labels, then ask generated gates to recover high-fidelity circuits.
exact_gates = centers_for_config(result.config.data, device=device)
exact_labels = center_names_for_config(result.config.data)

hidden_benchmarks = run_hidden_shallow_circuit_benchmark(
    exact_gates=exact_gates,
    exact_labels=exact_labels,
    generated_gates=local_gates,
    generated_labels=local_labels,
    n_targets=6,
    entangler="cz",
    n_random_candidates=100_000,
    top_k=5,
    seed=42,
)

print_hidden_shallow_circuit_benchmark(hidden_benchmarks)

hidden_label_grid_reports = {
    item.target.name: item.generated_label_grid_report for item in hidden_benchmarks
}
hidden_random_reports = {
    item.target.name: item.random_report for item in hidden_benchmarks
}

print("\nGenerated label-grid summary")
print_synthesis_summary(hidden_label_grid_reports)
plot_synthesis_fidelity_histograms(hidden_label_grid_reports, bins=60)

print("\nGenerated random-search summary")
print_synthesis_summary(hidden_random_reports)
plot_synthesis_fidelity_histograms(hidden_random_reports, bins=60)

In [ ]:
# Larger hidden shallow-circuit benchmark: summarize success rates over many hidden targets.
# This keeps only top candidates and best-fidelity aggregates, not every searched fidelity.
BENCHMARK_TARGETS = 50
BENCHMARK_RANDOM_CANDIDATES = 100_000

hidden_benchmark_suite = run_hidden_shallow_circuit_benchmark(
    exact_gates=exact_gates,
    exact_labels=exact_labels,
    generated_gates=local_gates,
    generated_labels=local_labels,
    n_targets=BENCHMARK_TARGETS,
    entangler="cz",
    n_random_candidates=BENCHMARK_RANDOM_CANDIDATES,
    top_k=5,
    seed=123,
    keep_fidelities=False,
)

print_hidden_shallow_circuit_summary(hidden_benchmark_suite)
plot_hidden_shallow_circuit_best_fidelities(hidden_benchmark_suite)

## Hidden two-entangler benchmark

Increase template depth to local - CZ - local - CZ - local. This uses random search over six generated local-gate slots, since exhaustive Clifford grids are too large at this depth.

In [ ]:
# Two-entangler hidden benchmark: a broader circuit family with six local slots.
TWO_ENTANGLER_TARGETS = 12
TWO_ENTANGLER_RANDOM_CANDIDATES = 200_000

two_entangler_benchmarks = run_hidden_two_entangler_circuit_benchmark(
    exact_gates=exact_gates,
    exact_labels=exact_labels,
    generated_gates=local_gates,
    generated_labels=local_labels,
    n_targets=TWO_ENTANGLER_TARGETS,
    entangler="cz",
    n_random_candidates=TWO_ENTANGLER_RANDOM_CANDIDATES,
    top_k=5,
    seed=321,
    keep_fidelities=False,
)

print_hidden_two_entangler_circuit_benchmark(two_entangler_benchmarks[:6])
print("\nTwo-entangler summary")
print_hidden_two_entangler_circuit_summary(two_entangler_benchmarks)
plot_hidden_two_entangler_best_fidelities(two_entangler_benchmarks)

In [ ]:
# Eta sweep: reuse the trained model to tune within-gate spread without retraining.
ETA_SWEEP = [0.3, 0.5, 0.7, 1.0]
eta_results = resample_experiment(result, ETA_SWEEP, device=device)

print_diagnostics_table({name: item.diagnostics for name, item in eta_results.items()})
print()
print_center_mass_table({name: item.diagnostics for name, item in eta_results.items()}, center_names=center_names)
conditional_eta = {
    name: item.conditional_diagnostics
    for name, item in eta_results.items()
    if item.conditional_diagnostics is not None
}
if conditional_eta:
    print()
    print_conditional_label_table(conditional_eta, center_names=center_names)

## Optional diagnostics

Eta sweeps and deeper per-center diagnostics are useful after the main benchmark output looks interesting.

In [ ]:
# Optional: slower, center-by-center gate diagnostics. Run only when the standard report looks interesting.
centers = centers_for_config(result.config.data, device=device)
deep_diagnostics = {
    name: diagnose_samples(
        samples,
        result.clean_reference,
        result.haar_reference,
        centers=centers,
        include_per_center=True,
    )
    for name, samples in {
        "deterministic": result.generated_deterministic,
        "stochastic": result.generated_stochastic,
    }.items()
}

print_diagnostics_table(deep_diagnostics)
print_per_center_table(deep_diagnostics, center_names=center_names)

In [ ]:
# Quick config comparison loop. Keep this to smoke/medium configs unless you want a long run.
for name in ["smoke", "smoke-gates", "smoke-gates-cond", "smoke-clifford-cond"]:
    cfg = get_experiment_config(name)
    cfg = replace(cfg, sample_count=512, reference_count=512)
    print("\n", cfg.name, cfg.data.kind, cfg.schedule.kind)
    quick = run_experiment(cfg, device=device)
    names = center_names_for_config(quick.config.data)
    print_diagnostics_table(quick.diagnostics)
    print_center_mass_table(quick.diagnostics, center_names=names)
    plot_experiment_report(quick)